# Two-Radius figure: accuracy vs $n$ (dims 256 / 1024)

Models: GCN, GAT, TransformerConv, SetTransformer; $n \in \{50,100,200\}$.
GCN and selected cells are hardcoded; remaining points from W&B project
`sro-base-sweep` (best `val_acc` over 200 epochs) plus `EXTRA_RUNS` where needed.
SetTransformer shown as 100% (saturates at lr $10^{-3}$). Metric: best val acc
within 200 epochs; single seed unless noted in sweep.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 12,
    "figure.titlesize": 14,
})
import wandb
import os

# Output dir: env FIGURE_OUTPUT_DIR, else ./results (run from bottleneck/)
RESULTS_DIR = os.environ.get("FIGURE_OUTPUT_DIR", "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

WANDB_PROJECT = "sro-base-sweep"
WANDB_ENTITY = None  # leave None to use your default entity
NS = [50, 100, 200]
DIMS = [256, 1024]
MODELS_TO_PLOT = ["GCN", "GAT", "TransformerConv", "SetTransformer"]

# GCN: fixed table from experiments
GCN_VALS = {
    (256, 50):  72.2611,
    (256, 100): 44.1272,
    (256, 200): 18.1272,
    (1024, 50):  83.1796,
    (1024, 100): 59.2571,
    (1024, 200): 36.17191,
}

# Points not covered by base sweep
# (best_val_acc as a fraction in [0, 1] -- will be converted to % later)
EXTRA_RUNS = pd.DataFrame([
    {"gnn": "TransformerConv", "dim": 1024, "n": 100, "best_val_acc": 0.9029},
    {"gnn": "GAT",             "dim": 1024, "n": 100, "best_val_acc": 0.99346},
])

# SetTransformer: 100% (saturated runs)
SETTRANSFORMER_VAL = 100.0  # %


## 1. Pull runs from W&B

Fetches every run in `sro-base-sweep`, reads the per-step `val_acc` history,
and records the best (max) value seen across the run. We then filter to the
configurations we care about.

If the cell hangs or errors, run `wandb login` once in a terminal first.

In [ ]:
import re

# Run names look like:
#   Transformer_two_connected_n50_K1_dim256_lr0.0002_VNFalse_h2_seed1
#   GAT_two_connected_n100_K1_dim1024_lr0.0002_VNFalse_h2_seed1
#   SetTransformer_two_connected_n50_K1_dim32_lr0.001_VNFalse_h4_seed1
NAME_RE = re.compile(
    r"^(?P<gnn>[A-Za-z]+)_two_connected"
    r"_n(?P<n>\d+)"
    r"_K(?P<K>\d+)"
    r"_dim(?P<dim>\d+)"
    r"_lr(?P<lr>[0-9.eE+-]+)"
    r"_VN(?P<vn>True|False)"
    r"_h(?P<heads>\d+)"
    r"_seed(?P<seed>\d+)"
)

api = wandb.Api(timeout=60)
project_path = f"{WANDB_ENTITY}/{WANDB_PROJECT}" if WANDB_ENTITY else WANDB_PROJECT
runs = list(api.runs(project_path))
print(f"Pulled {len(runs)} runs from {project_path}")

records = []
for run in runs:
    m = NAME_RE.match(run.name)
    if m:
        g = m.groupdict()
        parsed = {
            "gnn":   g["gnn"],
            "dim":   int(g["dim"]),
            "n":     int(g["n"]),
            "heads": int(g["heads"]),
            "lr":    float(g["lr"]),
            "vn":    (g["vn"] == "True"),
        }
    else:
        cfg = {**run.config}
        parsed = {
            "gnn":   cfg.get("gnn_type") or cfg.get("model_type"),
            "dim":   cfg.get("dim"),
            "n":     cfg.get("n") or cfg.get("start"),
            "heads": cfg.get("heads"),
            "lr":    cfg.get("lr"),
            "vn":    cfg.get("use_virtual_nodes"),
        }

    try:
        hist = run.history(keys=["val_acc"], pandas=True, samples=10000)
        best = float(hist["val_acc"].max()) if (not hist.empty and "val_acc" in hist) else float("nan")
    except Exception as e:
        print(f"  history fetch failed for {run.name}: {e}")
        best = float("nan")

    records.append({
        "run_name": run.name,
        "state":    run.state,
        **parsed,
        "best_val_acc": best,
    })

df_runs = pd.DataFrame(records)
n_clean = df_runs[["gnn", "dim", "n"]].notna().all(axis=1).sum()
print(f"Parsed {n_clean}/{len(df_runs)} runs cleanly from run_name")
df_runs[["run_name", "state", "gnn", "dim", "n", "heads", "lr", "vn", "best_val_acc"]]


## 2. Aggregate to a (gnn, dim, n) best-val-acc table

Normalise model names, drop runs that crashed/errored before logging anything,
and add the hardcoded extras for cells the W&B sweep did not cover.

In [ ]:
def normalise_gnn(g):
    if g is None:
        return None
    g = str(g)
    if g.lower() == "transformer":
        return "TransformerConv"
    return g

df = df_runs.copy()
df["gnn"] = df["gnn"].apply(normalise_gnn)
df = df.dropna(subset=["gnn", "dim", "n", "best_val_acc"])
df["dim"] = df["dim"].astype(int)
df["n"]   = df["n"].astype(int)

# Append hardcoded extras (n=100 dim=1024 GAT and TransformerConv).
df = pd.concat([df, EXTRA_RUNS], ignore_index=True)

# Best per (gnn, dim, n) -- handles duplicates / multiple LRs.
df_best = (
    df.groupby(["gnn", "dim", "n"])["best_val_acc"]
      .max()
      .reset_index()
)
df_best["best_val_acc_pct"] = df_best["best_val_acc"] * 100.0
df_best.sort_values(["gnn", "dim", "n"]).reset_index(drop=True)

## 3. Build the plotting matrix

Final table indexed by `(model, dim, n)` with values in **percent**. GCN comes
from the hardcoded dict; SetTransformer is hardcoded to 100%; everything else
from the W&B fetch above (with the two hardcoded extras).

In [ ]:
def lookup_pct(model, dim, n):
    """Return val_acc in % for (model, dim, n), or NaN if missing."""
    if model == "GCN":
        return GCN_VALS.get((dim, n), np.nan)
    if model == "SetTransformer":
        return SETTRANSFORMER_VAL
    sub = df_best[(df_best["gnn"] == model) & (df_best["dim"] == dim) & (df_best["n"] == n)]
    if sub.empty:
        return np.nan
    return float(sub["best_val_acc_pct"].iloc[0])

plot_table = pd.DataFrame(
    {(m, d): [lookup_pct(m, d, n) for n in NS] for m in MODELS_TO_PLOT for d in DIMS},
    index=NS,
)
plot_table.index.name = "n"
plot_table

## 4. Plot

Two-panel grouped bar chart matching the SRO Figure 2 layout. Colour palette
roughly mirrors the paper: GCN blue, GAT orange/yellow, TransformerConv pink,
SetTransformer purple/grey.

In [ ]:
COLORS = {
    "GCN":             "tab:blue",
    "GAT":             "tab:orange",
    "TransformerConv": "tab:red",
    "SetTransformer":  "tab:green",
}
# Display labels (with proper spacing) keyed by the internal model id.
DISPLAY_NAME = {
    "GCN":             "GCN",
    "GAT":             "GAT",
    "TransformerConv": "TransformerConv",
    "SetTransformer":  "Set Transformer",
}
PLOT_ORDER = ["GCN", "GAT", "TransformerConv", "SetTransformer"]

AXIS_LABEL_FS = 15
TICK_LABEL_FS = 13
TITLE_FS = 14

fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=False)

x = np.arange(len(NS), dtype=float)
n_models = len(PLOT_ORDER)
bar_width = 0.8 / n_models
offsets = (np.arange(n_models) - (n_models - 1) / 2.0) * bar_width

for ax, dim in zip(axes, DIMS):
    for i, model in enumerate(PLOT_ORDER):
        heights = [lookup_pct(model, dim, n) for n in NS]
        ax.bar(
            x + offsets[i], heights, width=bar_width,
            label=DISPLAY_NAME[model], color=COLORS[model],
            edgecolor="black", linewidth=0.4,
        )
    ax.set_xticks(x)
    ax.set_xticklabels(NS, fontsize=TICK_LABEL_FS)
    ax.set_xlabel("Number of nodes $n$", fontsize=AXIS_LABEL_FS)
    ax.set_title(f"Dim {dim}", fontsize=TITLE_FS)
    ax.set_ylim(0, 110)
    ax.set_ylabel("Accuracy [%]", fontsize=AXIS_LABEL_FS)
    ax.tick_params(axis="both", which="major", labelsize=TICK_LABEL_FS, labelleft=True)
    ax.grid(axis="y", linestyle=":", alpha=0.5)
    ax.set_axisbelow(True)

# Legend in the same order as the bars (left -> right within each cluster).
handles_by_label = {l: h for h, l in zip(*axes[0].get_legend_handles_labels())}
ordered_labels = [DISPLAY_NAME[m] for m in PLOT_ORDER]
fig.legend(
    [handles_by_label[l] for l in ordered_labels], ordered_labels,
    loc="upper center",
    ncol=len(ordered_labels), frameon=False, fontsize=13,
)

plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.savefig(f"{RESULTS_DIR}/figure2_reproduction.png", dpi=200, bbox_inches="tight")
plt.savefig(f"{RESULTS_DIR}/figure2_reproduction.pdf", bbox_inches="tight")
plt.show()
print("Saved -> results/figure2_reproduction.{png,pdf}")


## 5. Epochs to 90% accuracy vs. number of central nodes $k$

Bar chart of epochs-to-90%-validation-accuracy for the GAT K-sweep
(`dim=1024, n=100, h=2, lr=2e-4`). Values hardcoded from the run summaries.


In [ ]:
K_VALUES   = [1, 5, 10, 20]
EPOCHS_TO_90 = {1: 47, 5: 77, 10: 111, 20: 64}

AXIS_LABEL_FS = 15
TICK_LABEL_FS = 13

fig, ax = plt.subplots(figsize=(5, 5))
heights = [EPOCHS_TO_90[k] for k in K_VALUES]
ax.bar(
    [str(k) for k in K_VALUES], heights,
    color="tab:orange", edgecolor="black", linewidth=0.4,
    label="GAT",
)
ax.set_xlabel("Number of central nodes $k$", fontsize=AXIS_LABEL_FS)
ax.set_ylabel("Epochs to 90% accuracy", fontsize=AXIS_LABEL_FS)
ax.set_ylim(0, max(heights) * 1.15)
ax.tick_params(axis="both", which="major", labelsize=TICK_LABEL_FS)
ax.grid(axis="y", linestyle=":", alpha=0.5)
ax.set_axisbelow(True)
ax.legend(loc="upper right", frameon=True, fontsize=13)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/epochs_to_90_vs_k.png", dpi=200, bbox_inches="tight")
plt.savefig(f"{RESULTS_DIR}/epochs_to_90_vs_k.pdf", bbox_inches="tight")
plt.show()
print("Saved -> results/epochs_to_90_vs_k.{png,pdf}")


## 6. Average val_acc curves: VN strategies vs baselines

Curves averaged across the 3 seeds within each P1 group from the
`sro-vn-strategies` W&B project, plus the single Set Transformer run
from `sro-base-sweep`.

Groups plotted:
- `P1_DPW_d64`  -- Decoupled Probabilistic Wiring, d_router=64
- `P1_TPW`      -- Tied Probabilistic Wiring
- `P1_APW`      -- Adaptive Probabilistic Wiring (recompute per layer)
- `P1_no_vn`    -- GAT without virtual nodes
- Set Transformer (single run from `sro-base-sweep`)

Runs that early-stopped (val_acc >= 0.999) are extended with their last
value out to the longest run's epoch.


In [ ]:
VN_PROJECT     = "sro-vn-strategies"
BASE_PROJECT   = "sro-base-sweep"
GROUPS = {
    "P1_decoupled_d64":   "DPW (decoupled, d=64)",
    "P1_simple":          "TPW (tied)",
    "P1_dynamic":         "APW (adaptive)",
    "P1_no_vn":           "No VN (GAT)",
    "P1_set_transformer": "Set Transformer",
}

def fetch_val_acc_curve(run):
    """Return val_acc series indexed by the real `epoch` value logged by PL.

    Uses W&B's full (un-sampled) scan history so we get every val_acc point,
    indexed by the genuine `epoch` column instead of row position. This avoids
    the off-by-one caused by PL\'s sanity-check validation before epoch 0.
    """
    try:
        rows = list(run.scan_history(keys=["val_acc", "epoch"]))
    except Exception as e:
        print(f"  history fetch failed for {run.name}: {e}")
        return None
    if not rows:
        return None
    df = pd.DataFrame(rows).dropna(subset=["val_acc"])
    if df.empty:
        return None

    if "epoch" in df.columns and df["epoch"].notna().any():
        df = df.dropna(subset=["epoch"])
        df["epoch"] = df["epoch"].astype(int)
        # If multiple val_acc points exist for the same epoch (sanity check + real eval),
        # keep the *last* one logged for that epoch.
        df = df.drop_duplicates(subset=["epoch"], keep="last").sort_values("epoch")
        s = pd.Series(df["val_acc"].astype(float).values, index=df["epoch"].values)
    else:
        s = pd.Series(df["val_acc"].astype(float).values,
                      index=range(len(df)))
    s.index.name = "epoch"
    return s

# Early-stopped runs are averaged with missing epochs left as NaN.
PAD_MODE = "nan"

def pad_to_length(s, length, fill=PAD_MODE):
    """Reindex the series onto epochs 0..length-1, filling missing epochs."""
    full_index = pd.RangeIndex(0, length)
    s2 = s[s.index < length]
    s2 = s2.reindex(full_index)
    if fill == "last":
        last_observed_epoch = s.index.max()
        if pd.notna(last_observed_epoch):
            tail_mask = s2.index > last_observed_epoch
            s2.loc[tail_mask] = s.iloc[-1]
    return s2

def fetch_group_curves(project, group):
    runs = list(api.runs(project, filters={"group": group}))
    curves = []
    for r in runs:
        c = fetch_val_acc_curve(r)
        if c is not None:
            curves.append(c)
        else:
            print(f"  skipped (no val_acc): {r.name}")
    print(f"  group={group!r}: {len(runs)} runs, {len(curves)} with val_acc")
    return curves

print(f"Fetching group curves from project={VN_PROJECT!r}")
group_curves = {g: fetch_group_curves(VN_PROJECT, g) for g in GROUPS}

# Oracle (single run in P2D_oracle group, no averaging needed).
print(f"\nFetching oracle run from project={VN_PROJECT!r}")
oracle_runs = list(api.runs(VN_PROJECT, filters={"group": "P2D_oracle"}))
if not oracle_runs:
    raise RuntimeError("Could not find any runs in group P2D_oracle")
oracle_curve = fetch_val_acc_curve(oracle_runs[0])
print(f"  Oracle ({oracle_runs[0].name}): {len(oracle_curve)} epochs")

all_curves_for_max = [c for cs in group_curves.values() for c in cs] + [oracle_curve]
X_MAX_EPOCH = 80
max_len = min(max(int(c.index.max()) + 1 for c in all_curves_for_max), X_MAX_EPOCH)
print(f"\nMax run length capped at {max_len} epochs (X_MAX_EPOCH={X_MAX_EPOCH})")

group_means = {}
for g, curves in group_curves.items():
    if not curves:
        print(f"  WARNING: no curves for {g}")
        continue
    padded = [pad_to_length(c, max_len, fill="last").to_numpy() for c in curves]
    arr = np.vstack(padded)
    group_means[g] = pd.Series(np.nanmean(arr, axis=0), index=range(max_len))

# Oracle is a single run -- don't reindex (would introduce NaN gaps for
# sparsely logged runs). Just truncate to the visible epoch window.
oracle_curve_visible = oracle_curve[oracle_curve.index < max_len].sort_index()

# New label scheme: emphasise that everything except Set Transformer is GAT.
PLOT_ORDER = [
    ("group",  "P1_no_vn",         "GAT (No VNs)",                              "tab:blue",   "-"),
    ("oracle", None,               "GAT (VNs with identifier-matched wiring)",  "tab:gray",   ":"),
    ("group",  "P1_decoupled_d64", "GAT (VNs with DPW)",                        "tab:green",  "-"),
    ("group",  "P1_simple",        "GAT (VNs with TPW)",                        "tab:red",    "-"),
    ("group",  "P1_dynamic",       "GAT (VNs with APW)",                        "tab:orange", "-"),
    ("group",  "P1_set_transformer", "Set Transformer",                         "tab:purple", "-"),
]

AXIS_LABEL_FS = 15
TICK_LABEL_FS = 13

fig, ax = plt.subplots(figsize=(5, 5))

for kind, key, label, color, linestyle in PLOT_ORDER:
    if kind == "group":
        if key not in group_means:
            print(f"  [plot] skipping {label!r}: key {key!r} not in group_means")
            continue
        s = group_means[key]
    elif kind == "oracle":
        s = oracle_curve_visible
    else:
        continue
    n_valid = int(np.isfinite(s.values).sum())
    fvi, lvi = s.first_valid_index(), s.last_valid_index()
    print(f"  [plot] {label!r}: {n_valid} finite points (epochs {fvi}..{lvi})")
    ax.plot(s.index, s.values * 100,
            label=label, color=color, linewidth=2.2, linestyle=linestyle)

ax.set_xlabel("Epoch", fontsize=AXIS_LABEL_FS)
ax.set_ylabel("Accuracy [%]", fontsize=AXIS_LABEL_FS)
ax.set_ylim(0, 105)
ax.set_xlim(0, max_len - 1)
ax.tick_params(axis="both", which="major", labelsize=TICK_LABEL_FS)
ax.grid(linestyle=":", alpha=0.5)
ax.set_axisbelow(True)
ax.legend(loc="lower right", frameon=True, fontsize=13)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/p1_val_acc_curves.png", dpi=200, bbox_inches="tight")
plt.savefig(f"{RESULTS_DIR}/p1_val_acc_curves.pdf", bbox_inches="tight")
plt.show()
print("Saved -> results/p1_val_acc_curves.{png,pdf}")


## 7. Can VNs replace hidden dimension? -- dim=256 vs dim=1024 across $m$

Three panels for $m \in \{5, 10, 25\}$ virtual nodes, each comparing the
**dim=256** and **dim=1024** training curve under the *adaptive* (APW) strategy.

Single run per line (no averaging). For $m=25, dim=1024$ we use seed=1 from
group `P1_dynamic`, since `P2F_dynamic_d1024_m25` was intentionally skipped
in the sweep (the Part 1 seed=1 run is exactly that configuration).


In [ ]:
PANEL_M = [5, 10, 25]

def first_run_in(project, group, name_filter=None):
    runs = list(api.runs(project, filters={"group": group}))
    if name_filter is not None:
        runs = [r for r in runs if name_filter(r.name)]
    if not runs:
        raise RuntimeError(f"no runs in group={group!r}" + (" (after filter)" if name_filter else ""))
    return runs[0]

def extend_to(s, length):
    """Reindex onto epochs 0..length-1, forward-filling any gaps + trailing
    NaN with the last observed value. Lets early-stopped runs continue as
    a flat line at their final accuracy."""
    return s.sort_index().reindex(range(length)).ffill()

# Build {m: {"d256": curve, "d1024": curve}}
panel_curves = {}
for m in PANEL_M:
    d256_run  = first_run_in(VN_PROJECT, f"P2F_dynamic_d256_m{m}")
    if m == 25:
        # d1024_m25 not in P2F; use P1_dynamic seed=1 instead.
        d1024_run = first_run_in(VN_PROJECT, "P1_dynamic",
                                 name_filter=lambda n: "_seed1" in n)
        print(f"  m=25, dim=1024 -> {d1024_run.name} (from P1_dynamic seed=1)")
    else:
        d1024_run = first_run_in(VN_PROJECT, f"P2F_dynamic_d1024_m{m}")
    panel_curves[m] = {
        "d256":  fetch_val_acc_curve(d256_run),
        "d1024": fetch_val_acc_curve(d1024_run),
    }
    print(f"  m={m}: d256={len(panel_curves[m]['d256'])} pts, "
          f"d1024={len(panel_curves[m]['d1024'])} pts")

DIM_STYLE = {
    "d256":  ("dim=256",  "tab:orange"),
    "d1024": ("dim=1024", "tab:blue"),
}

AXIS_LABEL_FS = 15
TICK_LABEL_FS = 13
TITLE_FS = 14

fig, axes = plt.subplots(1, len(PANEL_M), figsize=(15, 5), sharey=True)

for ax, m in zip(axes, PANEL_M):
    for dim_key, (label, color) in DIM_STYLE.items():
        s = extend_to(panel_curves[m][dim_key], X_MAX_EPOCH)
        ax.plot(s.index, s.values * 100,
                label=label, color=color, linewidth=2.2)
    ax.set_title(fr"$m = {m}$ virtual nodes", fontsize=TITLE_FS)
    ax.set_xlabel("Epoch", fontsize=AXIS_LABEL_FS)
    ax.set_xlim(0, X_MAX_EPOCH)
    ax.set_ylim(0, 105)
    ax.tick_params(axis="both", which="major", labelsize=TICK_LABEL_FS)
    ax.grid(linestyle=":", alpha=0.5)
    ax.set_axisbelow(True)
    ax.legend(loc="lower right", frameon=True, fontsize=13)

axes[0].set_ylabel("Accuracy [%]", fontsize=AXIS_LABEL_FS)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/dim_vs_m_panels.png", dpi=200, bbox_inches="tight")
plt.savefig(f"{RESULTS_DIR}/dim_vs_m_panels.pdf", bbox_inches="tight")
plt.show()
print("Saved -> results/dim_vs_m_panels.{png,pdf}")
